# Segmentacija anomalija na poljoprivrednim snimcima

Ovaj notebook prikazuje tok projekta i najvaznije rezultate. Glavni kod se nalazi u Python fajlovima unutar foldera `src/`, a ovdje se koriste krace celije za pripremu okruzenja, ucitavanje podataka, prikaz primjera i poredjenje rezultata.

Cilj projekta je semanticka segmentacija anomalija na Agriculture-Vision datasetu. Za svaki piksel slike model treba da odredi da li pripada pozadini ili jednoj od anomalija: `double_plant`, `planter_skip`, `standing_water`, `waterway`, `weed_cluster`.


## 1. Pokretanje projekta u Google Colab okruzenju

Notebook je prilagodjen za rad u Google Colabu. Prvi korak je kloniranje GitHub repozitorijuma i ulazak u folder projekta.


In [ ]:
!git clone https://github.com/lukagrozdanic80-jpg/Agri-segmentation.git
%cd Agri-segmentation


Instaliraju se biblioteke koje projekat koristi. `opencv-python` je potreban zbog `cv2`, `albumentations` zbog augmentacija, a `segmentation-models-pytorch` za U-Net modele.


In [ ]:
!pip install -q -r requirements.txt


Provjera okruzenja pokazuje da li je GPU dostupan i da li su glavne biblioteke uspjesno instalirane.


In [ ]:
import torch
import cv2
import albumentations
import segmentation_models_pytorch as smp

print("torch:", torch.__version__)
print("cuda:", torch.cuda.is_available())
print("gpu:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU")
print("opencv:", cv2.__version__)
print("albumentations:", albumentations.__version__)
print("smp ok")


## 2. Preuzimanje i priprema dataseta

Koriscen je Agriculture-Vision 2021 dataset sa Hugging Face-a. Dataset se preuzima u `/content/data`, zatim se raspakuje u folder koji koristi ostatak notebooka.


In [ ]:
from pathlib import Path

DATA_DIR = Path("/content/data")
DATA_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:
from huggingface_hub import hf_hub_download

archive_path = hf_hub_download(
    repo_id="shi-labs/Agriculture-Vision",
    filename="Agriculture-Vision-2021.tar.gz",
    repo_type="dataset",
    local_dir=str(DATA_DIR),
)
print(archive_path)


In [ ]:
!mkdir -p /content/data/agriculture-vision
!tar --no-same-owner -xzf /content/data/Agriculture-Vision-2021.tar.gz -C /content/data/agriculture-vision


In [ ]:
DATA_ROOT = Path("/content/data/agriculture-vision/Agriculture-Vision-2021")
print("DATA_ROOT:", DATA_ROOT)
print("train RGB postoji:", (DATA_ROOT / "train/images/rgb").exists())
print("val RGB postoji:", (DATA_ROOT / "val/images/rgb").exists())


## 3. Klase u datasetu

U originalnom datasetu labele su odvojene po folderima. U projektu se koristi sest klasa: pozadina i pet ciljnih anomalija.

| ID | Klasa | Folder u datasetu | Znacenje |
|---:|---|---|---|
| 0 | `background` | nema poseban folder | piksel ne pripada ciljanoj anomaliji |
| 1 | `double_plant` | `labels/double_plant` | duplo zasadjena kultura |
| 2 | `planter_skip` | `labels/planter_skip` | preskocena sadnja |
| 3 | `standing_water` | `labels/water` | stajaca voda |
| 4 | `waterway` | `labels/waterway` | vodeni kanal ili vodeni tok |
| 5 | `weed_cluster` | `labels/weed_cluster` | grupa korova |


## 4. Ucitavanje jednog primjera

Sljedeci dio koristi `AgricultureVisionDataset` iz `src/dataset.py`. Loader vraca jedan `sample` koji sadrzi:

- `image`: RGB slika kao tensor oblika `[3, 512, 512]`
- `mask`: spojena segmentaciona maska oblika `[512, 512]`
- `valid_mask`: maska validnih piksela oblika `[512, 512]`
- `id`: naziv uzorka


In [ ]:
import sys
import numpy as np
import matplotlib.pyplot as plt

PROJECT_ROOT = Path("/content/Agri-segmentation")
if not PROJECT_ROOT.exists():
    PROJECT_ROOT = Path.cwd()

sys.path.append(str(PROJECT_ROOT / "src"))

from dataset import AgricultureVisionDataset, CLASS_NAMES

val_dataset = AgricultureVisionDataset(
    root=DATA_ROOT,
    split="val",
    image_size=512,
    use_nir=False,
    return_valid_mask=True,
    image_mean=[0.485, 0.456, 0.406],
    image_std=[0.229, 0.224, 0.225],
)

sample = val_dataset[0]
print("Broj validacionih uzoraka:", len(val_dataset))
print("ID uzorka:", sample["id"])
print("image:", sample["image"].shape, sample["image"].dtype)
print("mask:", sample["mask"].shape, sample["mask"].dtype)
print("valid_mask:", sample["valid_mask"].shape, sample["valid_mask"].dtype)
print("Klase prisutne u maski:", torch.unique(sample["mask"]).tolist())


Vizuelno se prikazuju ulazna RGB slika, spojena segmentaciona maska i valid maska. Ovo je najjednostavnija provjera da dataset loader radi ispravno.


In [ ]:
def denormalize_image(image_tensor, mean, std):
    image = image_tensor.permute(1, 2, 0).cpu().numpy()
    image = image * np.array(std) + np.array(mean)
    return np.clip(image, 0, 1)

image = denormalize_image(sample["image"], [0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
mask = sample["mask"].cpu().numpy()
valid_mask = sample["valid_mask"].cpu().numpy()

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes[0].imshow(image)
axes[0].set_title("RGB slika")
axes[1].imshow(mask, cmap="tab10", vmin=0, vmax=len(CLASS_NAMES)-1)
axes[1].set_title("Segmentaciona maska")
axes[2].imshow(valid_mask, cmap="gray")
axes[2].set_title("Valid maska")

for ax in axes:
    ax.axis("off")

plt.tight_layout()
plt.show()


## 5. Preprocesiranje u loaderu

Ucitavanje podataka nije samo citanje slike iz foldera. `AgricultureVisionDataset` radi vise koraka prije nego sto podatak dodje do modela.

### 5.1 Ucitavanje RGB slike

RGB slika se cita iz foldera `images/rgb` i konvertuje iz BGR u RGB format, jer OpenCV podrazumijevano ucitava BGR.

```python
image = cv2.imread(str(rgb_path), cv2.IMREAD_COLOR)
image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
```

### 5.2 Ucitavanje maski po klasama

Svaka anomalija ima poseban folder sa binarnim maskama. Ako je piksel bijel u toj maski, taj piksel pripada toj klasi.

```python
class_mask = cv2.imread(str(label_path), cv2.IMREAD_GRAYSCALE)
mask[class_mask > 0] = class_id
```

### 5.3 Spajanje maski u jednu multiclass masku

Model ne dobija pet odvojenih maski, nego jednu masku u kojoj svaki piksel ima ID klase.

```python
mask = np.zeros((height, width), dtype=np.int64)
mask[class_mask > 0] = class_id
```

### 5.4 Valid maska

Valid maska oznacava piksele koji se koriste u loss funkciji i mIoU racunanju. Nevalidni pikseli se ignorisu.

```python
valid_mask = cv2.imread(str(valid_mask_path), cv2.IMREAD_GRAYSCALE) > 0
```

### 5.5 Normalizacija slike

Slika se normalizuje ImageNet vrijednostima jer su encoderi prethodno trenirani na ImageNet podacima.

```python
image = (image / 255.0 - mean) / std
```

### 5.6 Pretvaranje u tensore

Na kraju se NumPy nizovi pretvaraju u PyTorch tensore koje model moze koristiti.

```python
image = torch.from_numpy(image).permute(2, 0, 1).float()
mask = torch.from_numpy(mask).long()
valid_mask = torch.from_numpy(valid_mask).bool()
```


## 6. Modeli

| Eksperiment | Model | Podesavanje | Razlog testiranja |
|---|---|---|---|
| Baseline | U-Net + ResNet-50 | bez augmentacija | osnovni rezultat za poredjenje |
| Augmentacije | U-Net + ResNet-50 | flip, rotacija, brightness/contrast, noise | provjera da li augmentacije poboljsavaju generalizaciju |
| Jaci CNN encoder | U-Net + EfficientNet-B3 | iste augmentacije | poredjenje ResNet-50 i EfficientNet-B3 encodera |
| Transformer pristup | DINOv3 ViT-S/16 | pretrained DINOv3 backbone | provjera da li transformer reprezentacije daju bolju segmentaciju |


## 7. Trening podesavanja

Trening je implementiran u `src/train.py`, a podesavanja se citaju iz YAML config fajlova u folderu `configs/`.

Najvaznije stavke su:

| Stavka | Koriscena vrijednost | Uloga |
|---|---|---|
| Loss | Cross-Entropy + Dice Loss | kaznjava pogresnu klasu po pikselu i mjeri preklapanje maski |
| Optimizer | AdamW | Adam optimizer sa weight decay regularizacijom |
| Scheduler | CosineAnnealingLR | postepeno smanjuje learning rate tokom treninga |
| Metrika | mean IoU | mjeri preklapanje predikcije i stvarne maske po klasama |
| Checkpoint | najbolji model po validation mIoU | cuva se model koji najbolje radi na validacionom skupu |

Primjer pokretanja baseline treninga:


In [ ]:
baseline_command = """python src/train.py \
  --config configs/baseline_unet_resnet50.yaml \
  --data-root /content/data/agriculture-vision/Agriculture-Vision-2021 \
  --output-dir outputs_baseline \
  --epochs 2"""
print(baseline_command)


Primjer pokretanja DINOv3 treninga:


In [ ]:
dino_command = """python src/train.py \
  --config configs/dino_v3_vits16_seg.yaml \
  --data-root /content/data/agriculture-vision/Agriculture-Vision-2021 \
  --output-dir outputs_dino_v3_full_1ep \
  --epochs 1"""
print(dino_command)


## 8. Rezultati

| Eksperiment | Best validation mIoU |
|---|---:|
| U-Net + ResNet-50 baseline | 0.4892 |
| U-Net + ResNet-50 light aug | 0.4751 |
| U-Net + ResNet-50 full aug | 0.5331 |
| Weighted loss partial | 0.4007 |
| Soft weighted loss partial | 0.4107 |
| ImageNet norm partial | 0.4509 |
| U-Net + EfficientNet-B3 aug | 0.5587 |
| DINOv3 partial + full validation | 0.7169 |
| DINOv3 full 1 epoch | 0.6943 |

Najbolji rezultat dao je DINOv3. Najbolji CNN rezultat dao je U-Net + EfficientNet-B3.


Isti rezultati su prikazani i grafifki radi lakseg poredjenja.


In [ ]:
results = {
    "U-Net + ResNet-50 baseline": 0.4892,
    "U-Net + ResNet-50 light aug": 0.4751,
    "U-Net + ResNet-50 full aug": 0.5331,
    "Weighted loss partial": 0.4007,
    "Soft weighted loss partial": 0.4107,
    "ImageNet norm partial": 0.4509,
    "U-Net + EfficientNet-B3 aug": 0.5587,
    "DINOv3 partial + full val": 0.7169,
    "DINOv3 full 1 epoch": 0.6943,
}

labels = list(results.keys())
values = list(results.values())
order = np.argsort(values)

plt.figure(figsize=(10, 6))
plt.barh(np.array(labels)[order], np.array(values)[order])
plt.xlabel("Best validation mIoU")
plt.title("Poredjenje eksperimenata")
plt.xlim(0, 0.8)
plt.grid(axis="x", alpha=0.25)
plt.tight_layout()
plt.show()


## 9. Evaluacija sacuvanog checkpointa

Evaluacija ucitava najbolji checkpoint, prolazi kroz validacioni skup, racuna validation loss i mIoU, i cuva nekoliko slika predikcija.


In [ ]:
evaluate_command = """python src/evaluate.py \
  --checkpoint outputs_dino_v3_full_1ep/best_dino_v3_vits16.pth \
  --config configs/dino_v3_vits16_seg.yaml \
  --data-root /content/data/agriculture-vision/Agriculture-Vision-2021 \
  --output-dir outputs_dino_v3_full_1ep/evaluation \
  --save-samples 8"""
print(evaluate_command)


Ako su evaluacione slike vec sacuvane u `output_dir/evaluation`, mogu se prikazati direktno u notebooku.


In [ ]:
from PIL import Image

EVAL_DIR = PROJECT_ROOT / "outputs_dino_v3_full_1ep" / "evaluation"
image_paths = sorted(EVAL_DIR.glob("*.png")) if EVAL_DIR.exists() else []

print("Broj pronadjenih evaluacionih slika:", len(image_paths))

for image_path in image_paths[:4]:
    plt.figure(figsize=(10, 6))
    plt.imshow(Image.open(image_path))
    plt.title(image_path.name)
    plt.axis("off")
    plt.show()


## 10. Zakljucak

U projektu je napravljen kompletan pipeline za semanticku segmentaciju: dataset loader, spajanje maski, augmentacije, modeli, trening, validacija, mIoU metrika i cuvanje najboljih checkpointa.

Baseline U-Net + ResNet-50 daje pocetnu vrijednost za poredjenje. Augmentacije poboljsavaju ResNet-50 rezultat, EfficientNet-B3 daje najbolji CNN rezultat, a DINOv3 daje najbolji ukupni rezultat.
